# CY-Bench Stage 1 → 2 → 3 消融实验（强制单一候选）

与 `run_stage123_rag_5repeats.ipynb` 完全相同的流程（Stage 1 推荐/校准筛选 → 正式五模型五折预测 → 汇总；
Stage 2 CN 数据导出与 TabPFN 对比；Stage 3 风险诊断/恢复情景/LLM 建议），唯一区别是把两个 LLM
advisor 的 Stage 1 特征候选各**固定为一个**：

- `qwen_base` → 仅使用 `extreme_heat_drought_syndrome` 配方（复用原 CN 推荐中 `sample1_extreme_heat_drought_syndrome` 的 config，各国相同）；
- `qwen_rag`  → 仅使用 Historical regime（各国原 RAG 推荐中的 `rag1_rag_historical_regime`）。

运行范围：国家 CN、ZA、MX、PT；模型与重复数与原文一致。所有结果写入**新目录**
`cybench/output/stages_abl_heat_hist/`，**不会覆盖**原始 `cybench/output/stages/`。

请使用 **specllm** 内核运行。每个长任务实时输出并保存日志；中断后重跑对应单元即可从已有产物继续。

In [5]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys
import time

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'cybench').is_dir():
    raise RuntimeError(f'请从项目根目录打开 Notebook；当前目录为 {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)

COUNTRIES = ['CN', 'ZA', 'MX', 'PT']
COUNTRY_ARG = ','.join(COUNTRIES)
ADVISORS = ['hardcode', 'qwen_base', 'qwen_rag']
ADVISOR_ARG = ','.join(ADVISORS)
MODELS = ['ridge', 'xgboost', 'svr', 'cnn1d', 'transformer_flat']
REPEATS = 5
BASE_SEED = 42
DEVICE = 'cuda'

BASE = PROJECT_ROOT / 'cybench' / 'output' / 'stages_abl_heat_hist'
STAGE_ROOT = BASE
STAGE1 = STAGE_ROOT / 'stage1_feature_engineering'
STAGE2 = STAGE_ROOT / 'stage2_tabpfn_transfer' / 'evaluation'
STAGE3 = STAGE_ROOT / 'stage3_advisory'
STAGE1_REC = STAGE1 / 'recommendations'
STAGE1_RES = STAGE1 / 'results'
STAGE2_CN = STAGE_ROOT / 'stage2_tabpfn_transfer' / 'cn_finetuning'
ORIG_STAGE1 = PROJECT_ROOT / 'cybench' / 'output' / 'stages' / 'stage1_feature_engineering'
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = PROJECT_ROOT / 'cybench' / 'output' / 'logs' / f'notebook_stage123_abl_heat_hist_{RUN_ID}'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 消融实验不调用 LLM 重新生成推荐：候选在本 notebook 中直接固定写入新 raw 目录。
REGENERATE_LLM = False

print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)
print('Countries:', COUNTRY_ARG)
print('Repeats:', REPEATS, 'Seeds:', [BASE_SEED + 1009 * i for i in range(REPEATS)])
print('Regenerate missing LLM recommendations:', REGENERATE_LLM)
if 'specllm' not in sys.executable.lower():
    raise RuntimeError('当前不是 specllm Python 内核，请切换内核后重新运行。')

Python: d:\Anaconda\envs\specllm\python.exe
Project: C:\Users\35454\Downloads\AgML-CY-Bench-main
Countries: CN,ZA,MX,PT
Repeats: 5 Seeds: [42, 1051, 2060, 3069, 4078]
Regenerate missing LLM recommendations: False


## 实验设计说明

- Stage 2/3 沿用 `qwen_rag`（现被固定为 Historical regime）的 Stage 1 特征集；`hardcode` 基线不变。
- Stage 1 仍对 hardcode / qwen_base / qwen_rag × 5 模型 × 5 次重复做正式预测与汇总。
- 预检单元会先用强制配方在每个国家构建特征表，避免长任务中途因特征不兼容而失败。

In [6]:
# 强制候选写入 + 特征可构建预检
# qwen_base -> extreme_heat_drought_syndrome（复用原 CN 推荐的配方，各国相同）
# qwen_rag  -> rag_historical_regime（各国原 RAG 推荐中的 Historical regime slot）
# 只写新目录 STAGE1_REC/raw/...，不触碰原推荐目录。
import copy
import json as _json
from pathlib import Path

FORCED_QWEN_BASE_CANDIDATE = 'sample1_extreme_heat_drought_syndrome'
FORCED_QWEN_RAG_CANDIDATE = 'rag1_rag_historical_regime'
ORIG_RAW = ORIG_STAGE1 / 'recommendations' / 'raw'


def _read_json(path):
    return _json.loads(Path(path).read_text(encoding='utf-8-sig'))


def _write_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')


# 原 CN qwen_base 推荐中的 extreme_heat_drought_syndrome 候选（config 自包含、跨国家可复用）。
cn_base_raw = _read_json(ORIG_RAW / 'maize_CN' / 'recommendation_qwen_base.json')
cn_matches = [c for c in cn_base_raw['candidates'] if c['candidate_id'] == FORCED_QWEN_BASE_CANDIDATE]
assert len(cn_matches) == 1, f'原 CN qwen_base 推荐缺少 {FORCED_QWEN_BASE_CANDIDATE}'
forced_base_candidate = copy.deepcopy(cn_matches[0])

forced_reports = {}
for country in COUNTRIES:
    orig_rag = _read_json(ORIG_RAW / f'maize_{country}' / 'recommendation_qwen_rag.json')
    rag_cands = [c for c in orig_rag['candidates'] if c['candidate_id'] == FORCED_QWEN_RAG_CANDIDATE]
    assert len(rag_cands) == 1, f'{country} qwen_rag 原始推荐缺少 {FORCED_QWEN_RAG_CANDIDATE}'
    forced_reports[country] = {
        'qwen_rag': {**orig_rag, 'candidates': rag_cands},
        'qwen_base': {'candidates': [copy.deepcopy(forced_base_candidate)],
                      'country': country, 'advisor': 'qwen_base'},
    }
    for advisor, report in forced_reports[country].items():
        _write_json(report, STAGE1_REC / 'raw' / f'maize_{country}' / f'recommendation_{advisor}.json')

# 预检：确保两个强制配方都能在目标国家构建出特征表。
from cybench.stages.data import (
    build_feature_table,
    feature_columns,
    load_country_dataset,
    merge_features_labels,
)

for country in COUNTRIES:
    dataset = load_country_dataset('maize', country)
    configs = {
        'qwen_base': forced_base_candidate['config'],
        'qwen_rag': forced_reports[country]['qwen_rag']['candidates'][0]['config'],
    }
    for advisor, config in configs.items():
        frame = merge_features_labels(build_feature_table(dataset, config), dataset)
        print(country, advisor, 'feature_columns=', len(feature_columns(frame)))
print('强制候选已写入：', STAGE1_REC / 'raw')


CN qwen_base feature_columns= 100
CN qwen_rag feature_columns= 100
ZA qwen_base feature_columns= 112
ZA qwen_rag feature_columns= 112
MX qwen_base feature_columns= 126
MX qwen_rag feature_columns= 126
PT qwen_base feature_columns= 126
PT qwen_rag feature_columns= 126
强制候选已写入： C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage1_feature_engineering\recommendations\raw


In [7]:
def run_module(step_name, module, args):
    cmd = [sys.executable, '-u', '-m', module, *[str(value) for value in args]]
    log_path = LOG_DIR / f'{step_name}.log'
    print('\n' + '=' * 90)
    print(step_name)
    print(' '.join(cmd))
    print('Log:', log_path)
    started = time.time()
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(
            cmd, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        return_code = process.wait()
    elapsed = time.time() - started
    state = {'step': step_name, 'module': module, 'command': cmd, 'return_code': return_code, 'elapsed_seconds': elapsed}
    (LOG_DIR / f'{step_name}.state.json').write_text(json.dumps(state, indent=2, ensure_ascii=False), encoding='utf-8')
    if return_code != 0:
        raise RuntimeError(f'{step_name} 失败，返回码 {return_code}；查看 {log_path}')
    print(f'完成：{step_name}，耗时 {elapsed / 60:.1f} 分钟')
    return state

def require_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


## Stage 1：推荐、校准期候选筛选

In [8]:
stage1_prepare_args = [
    '--countries', COUNTRY_ARG, '--advisors', ADVISOR_ARG,
    '--direct-candidates', '--selection-repeats', REPEATS,
    '--base-seed', BASE_SEED, '--resume',
    '--output-dir', STAGE1_REC,
]
if REGENERATE_LLM:
    stage1_prepare_args.append('--regenerate-llm')
run_module('01_stage1_prepare', 'cybench.stages.stage1_feature_engineering.prepare', stage1_prepare_args)


01_stage1_prepare
d:\Anaconda\envs\specllm\python.exe -u -m cybench.stages.stage1_feature_engineering.prepare --countries CN,ZA,MX,PT --advisors hardcode,qwen_base,qwen_rag --direct-candidates --selection-repeats 5 --base-seed 42 --resume --output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage1_feature_engineering\recommendations
Log: C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\logs\notebook_stage123_abl_heat_hist_20260912_160855\01_stage1_prepare.log
CN qwen_base: prepared checkpoint loaded
CN qwen_rag: prepared checkpoint loaded
ZA qwen_base: prepared checkpoint loaded
ZA qwen_rag: prepared checkpoint loaded
MX qwen_base: prepared checkpoint loaded
MX qwen_rag: prepared checkpoint loaded
PT qwen_base: prepared checkpoint loaded
PT qwen_rag: prepared checkpoint loaded
完成：01_stage1_prepare，耗时 0.1 分钟


{'step': '01_stage1_prepare',
 'module': 'cybench.stages.stage1_feature_engineering.prepare',
 'command': ['d:\\Anaconda\\envs\\specllm\\python.exe',
  '-u',
  '-m',
  'cybench.stages.stage1_feature_engineering.prepare',
  '--countries',
  'CN,ZA,MX,PT',
  '--advisors',
  'hardcode,qwen_base,qwen_rag',
  '--direct-candidates',
  '--selection-repeats',
  '5',
  '--base-seed',
  '42',
  '--resume',
  '--output-dir',
  'C:\\Users\\35454\\Downloads\\AgML-CY-Bench-main\\cybench\\output\\stages_abl_heat_hist\\stage1_feature_engineering\\recommendations'],
 'return_code': 0,
 'elapsed_seconds': 3.110807418823242}

## Stage 1：外层五次正式预测与汇总

In [ ]:
run_module(
    '02_stage1_run', 'cybench.stages.stage1_feature_engineering.run',
    ['--countries', COUNTRY_ARG, '--advisors', ADVISOR_ARG, '--models', ','.join(MODELS),
     '--repeats', REPEATS, '--base-seed', BASE_SEED,
     '--resume',
     '--config-dir', STAGE1_REC / 'selected_configs', '--output-dir', STAGE1_RES],
)
run_module('03_stage1_summarize', 'cybench.stages.stage1_feature_engineering.summarize',
    ['--input', STAGE1_RES / 'metrics_all.csv', '--output-dir', STAGE1_RES])
display(pd.read_csv(require_file(STAGE1 / 'results' / 'metrics_summary.csv')).query("split == 'test'"))

## Stage 2：RAG 特征上的原始/农业继续训练 TabPFN

In [6]:
run_module('04_stage2_prepare_cn', 'cybench.stages.stage2_tabpfn_transfer.prepare_cn_data',
    ['--crop', 'maize', '--stage1-output-dir', STAGE1, '--output-dir', STAGE2_CN])
run_module(
    '05_stage2_run', 'cybench.stages.stage2_tabpfn_transfer.run',
    ['--countries', COUNTRY_ARG, '--device', DEVICE, '--repeats', REPEATS,
     '--base-seed', BASE_SEED, '--resume',
     '--stage1-output-dir', STAGE1, '--output-dir', STAGE2],
)
display(pd.read_csv(require_file(STAGE2 / 'metrics_summary.csv')).query("split == 'test'"))


04_stage2_prepare_cn
d:\Anaconda\envs\specllm\python.exe -u -m cybench.stages.stage2_tabpfn_transfer.prepare_cn_data --crop maize --stage1-output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage1_feature_engineering --output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage2_tabpfn_transfer\cn_finetuning
Log: C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\logs\notebook_stage123_abl_heat_hist_20260912_142554\04_stage2_prepare_cn.log
{
  "schema_version": 1,
  "created_at_utc": "2026-09-12T06:49:21.003478+00:00",
  "crop": "maize",
  "source_country": "CN",
  "feature_protocol": "Stage 1 Base-Qwen structured-CSV RAG features",
  "feature_columns": [
    "awc",
    "bulk_density",
    "max_cum_gdd1",
    "max_cum_gdd2",
    "max_cum_gdd3",
    "max_cum_gdd4",
    "max_cum_gdd5",
    "max_cum_gdd6",
    "max_cum_gdd7",
    "max_cum_prec1",
    "max_cum_prec2",
    "max_cum_prec3",
    "max_cum_pre

,country,model,checkpoint_sha256,test_years,n_features,split,mape_mean,normalized_rmse_mean,r_mean,r2_mean,kge_mean,mape_variance,normalized_rmse_variance,r_variance,r2_variance,kge_variance
0,CN,tabpfn_base,311ce18d97e9533d8585eaadafe040fbdd8070533209ed...,2020;2021;2022,100,test,0.05920,7.04646,0.95498,0.86334,0.92554,4.200000e-07,0.013781,0.000003,0.000021,0.000004
2,CN,tabpfn_cn_finetuned,53119ca48958a7b7eb5ca06752eecc2348f0b2ffa58d45...,2020;2021;2022,100,test,0.05764,6.89936,0.95492,0.86902,0.92412,4.030000e-07,0.014618,0.000003,0.000021,0.000004
4,MX,tabpfn_base,311ce18d97e9533d8585eaadafe040fbdd8070533209ed...,2022,126,test,0.31746,28.62632,0.94346,0.85982,0.77788,2.709080e-04,0.853173,0.000024,0.000081,0.000008
6,MX,tabpfn_cn_finetuned,53119ca48958a7b7eb5ca06752eecc2348f0b2ffa58d45...,2022,126,test,0.31396,28.30110,0.94646,0.86304,0.77552,1.372630e-04,0.378093,0.000012,0.000036,0.000010
8,PT,tabpfn_base,311ce18d97e9533d8585eaadafe040fbdd8070533209ed...,2016;2017;2018;2019;2020,126,test,0.20228,25.61578,0.87456,0.44802,0.68618,2.244700e-05,0.306641,0.000018,0.000575,0.000088
10,PT,tabpfn_cn_finetuned,53119ca48958a7b7eb5ca06752eecc2348f0b2ffa58d45...,2016;2017;2018;2019;2020,126,test,0.19670,25.07390,0.88300,0.47116,0.68914,1.596500e-05,0.252116,0.000024,0.000452,0.000084
12,ZA,tabpfn_base,311ce18d97e9533d8585eaadafe040fbdd8070533209ed...,2017;2018;2019;2020;2021;2022,112,test,0.16198,18.34962,0.94202,0.65702,0.72288,3.717000e-06,0.057406,0.000014,0.000081,0.000015
14,ZA,tabpfn_cn_finetuned,53119ca48958a7b7eb5ca06752eecc2348f0b2ffa58d45...,2017;2018;2019;2020;2021;2022,112,test,0.15682,17.83746,0.94162,0.67592,0.73570,2.117000e-06,0.030693,0.000023,0.000041,0.000059


## Stage 3：风险诊断、受约束恢复情景与 LLM 建议

Stage 3 只运行 `qwen_rag` 分支。`qwen_base`（无检索退化参考）与 `deterministic_policy`（确定性策略）
不再生成，相关旧产物目录需另行清理。

In [7]:
run_module(
    '06_stage3_run', 'cybench.stages.stage3_advisory.run',
    ['--countries', COUNTRY_ARG, '--repeats', REPEATS, '--base-seed', BASE_SEED,
     '--advisors', 'qwen_rag',
     '--stage1-output-dir', STAGE1, '--stage2-output-dir', STAGE2, '--output-dir', STAGE3],
)


06_stage3_run
d:\Anaconda\envs\specllm\python.exe -u -m cybench.stages.stage3_advisory.run --countries CN,ZA,MX,PT --repeats 5 --base-seed 42 --advisors qwen_rag --stage1-output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage1_feature_engineering --stage2-output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage2_tabpfn_transfer\evaluation --output-dir C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage3_advisory
Log: C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\logs\notebook_stage123_abl_heat_hist_20260912_142554\06_stage3_run.log
  LLM advisor 'stress_scenario_policy': loaded from cache
  LLM advisor 'stress_adaptation_advice': generating recommendation...
Loading LLM from C:\Users\35454\Downloads\AgML-CY-Bench-main\LLM\Qwen3.5-9B...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Exception in thread Thread-5384 (_readerthread):
Traceback 

{'step': '06_stage3_run',
 'module': 'cybench.stages.stage3_advisory.run',
 'command': ['d:\\Anaconda\\envs\\specllm\\python.exe',
  '-u',
  '-m',
  'cybench.stages.stage3_advisory.run',
  '--countries',
  'CN,ZA,MX,PT',
  '--repeats',
  '5',
  '--base-seed',
  '42',
  '--advisors',
  'qwen_rag',
  '--stage1-output-dir',
  'C:\\Users\\35454\\Downloads\\AgML-CY-Bench-main\\cybench\\output\\stages_abl_heat_hist\\stage1_feature_engineering',
  '--stage2-output-dir',
  'C:\\Users\\35454\\Downloads\\AgML-CY-Bench-main\\cybench\\output\\stages_abl_heat_hist\\stage2_tabpfn_transfer\\evaluation',
  '--output-dir',
  'C:\\Users\\35454\\Downloads\\AgML-CY-Bench-main\\cybench\\output\\stages_abl_heat_hist\\stage3_advisory'],
 'return_code': 0,
 'elapsed_seconds': 1234.7075955867767}

## 完整性审计：每个模型必须有五次 train/test

In [8]:
EXPECTED_REPEATS = set(range(1, REPEATS + 1))
EXPECTED_SPLITS = {'train', 'test'}

def audit_prediction_files(paths, label, expected_count):
    paths = list(paths)
    problems = []
    for path in paths:
        frame = pd.read_csv(path)
        repeats = set(pd.to_numeric(frame.get('repeat'), errors='coerce').dropna().astype(int))
        splits = set(frame.get('split', pd.Series(dtype=str)).dropna().astype(str))
        if repeats != EXPECTED_REPEATS or splits != EXPECTED_SPLITS:
            problems.append({'path': str(path), 'repeats': sorted(repeats), 'splits': sorted(splits)})
    result = {'stage': label, 'files': len(paths), 'expected_files': expected_count, 'invalid_files': len(problems)}
    print(result)
    if len(paths) != expected_count or problems:
        raise AssertionError({'summary': result, 'problems': problems[:10]})
    return result

audit = []
audit.append(audit_prediction_files(
    STAGE1.glob('results/maize_*/*/predictions_*.csv'), 'stage1',
    len(COUNTRIES) * len(ADVISORS) * len(MODELS),
))
audit.append(audit_prediction_files(
    STAGE2.glob('maize_*/predictions_*.csv'), 'stage2', len(COUNTRIES) * 2,
))
audit.append(audit_prediction_files(
    STAGE3.glob('maize_*/model_predictions.csv'), 'stage3', len(COUNTRIES),
))

required_summaries = [
    STAGE1 / 'results' / 'metrics_all.csv', STAGE1 / 'results' / 'metrics_summary.csv',
    STAGE2 / 'metrics_all.csv', STAGE2 / 'metrics_summary.csv',
]
required_summaries += [STAGE3 / f'maize_{country}' / 'model_validation_repeats.csv' for country in COUNTRIES]
required_summaries += [STAGE3 / f'maize_{country}' / 'model_validation_summary.csv' for country in COUNTRIES]
for path in required_summaries:
    require_file(path)

completion = {
    'completed_at': datetime.now().isoformat(), 'countries': COUNTRIES,
    'advisors': ADVISORS, 'models': MODELS, 'repeats': REPEATS,
    'base_seed': BASE_SEED, 'audit': audit,
}
completion_path = STAGE_ROOT / 'stage123_abl_heat_hist_completion.json'
completion_path.write_text(json.dumps(completion, indent=2, ensure_ascii=False), encoding='utf-8')
print('全部完成并通过审计：', completion_path)

{'stage': 'stage1', 'files': 60, 'expected_files': 60, 'invalid_files': 0}
{'stage': 'stage2', 'files': 8, 'expected_files': 8, 'invalid_files': 0}
{'stage': 'stage3', 'files': 4, 'expected_files': 4, 'invalid_files': 0}
全部完成并通过审计： C:\Users\35454\Downloads\AgML-CY-Bench-main\cybench\output\stages_abl_heat_hist\stage123_abl_heat_hist_completion.json
